# 1. Application Security Design

SC-100 asks: **"Design a secure application lifecycle — from threat modeling through DevSecOps to production workload protection."**

## Setup

```bash
cd security/sc-100/04-applications-and-data
uv sync
uv run python -m ipykernel install --user --name=sc-100 --display-name="SC-100 (Python)"
```
Then select the **SC-100 (Python)** kernel in VS Code's kernel picker (top-right).

If the kernel doesn't appear, reload the VS Code window (`Cmd+Shift+P` → "Reload Window").

## Threat Modeling with STRIDE

Threat modeling is **the most important security design activity**. It should happen early in the design phase, not after deployment.

### STRIDE categories:

| Letter | Threat | Property violated | Example |
|--------|--------|------------------|---------|
| **S** | Spoofing | Authentication | Attacker impersonates a user or service |
| **T** | Tampering | Integrity | Attacker modifies data in transit or at rest |
| **R** | Repudiation | Non-repudiation | User denies performing an action, no audit trail |
| **I** | Information Disclosure | Confidentiality | Sensitive data exposed (PII, secrets, tokens) |
| **D** | Denial of Service | Availability | Application becomes unavailable |
| **E** | Elevation of Privilege | Authorization | User gains unauthorized access to higher privilege |

### Threat modeling process:

```
1. DEFINE SCOPE
   └── What are we modeling? (system, feature, data flow)
         │
2. CREATE DATA FLOW DIAGRAM (DFD)
   └── Identify: processes, data stores, external entities, data flows, trust boundaries
         │
3. IDENTIFY THREATS (STRIDE per element)
   └── For each element in the DFD, ask: what could go wrong?
         │
4. RATE AND PRIORITIZE (DREAD or risk matrix)
   └── Likelihood × Impact = Risk priority
         │
5. DETERMINE MITIGATIONS
   └── Design controls: what Azure/Microsoft features address each threat?
         │
6. VALIDATE
   └── Review with dev team, update as architecture evolves
```

In [ ]:
import json

# ===================================================================
# THREAT MODELING EXERCISE
# Apply STRIDE to a real-world e-commerce application
# ===================================================================

APP_DESCRIPTION = """
APPLICATION: Tailspin Toys Online Store

Architecture:
  User (browser) → Azure Front Door → AKS (API) → Azure SQL (data)
                                         ↓
                                    Key Vault (secrets)
                                         ↓
                                   Storage (product images)
                                         ↓
                                  Payment Gateway (3rd party)

Trust boundaries:
  TB1: Internet → Azure Front Door (public to private)
  TB2: AKS → Azure SQL (app to data)
  TB3: AKS → Payment Gateway (internal to external 3rd party)
"""

STRIDE_ANALYSIS = [
    {
        'element': 'User → Front Door (TB1)',
        'threat': 'Spoofing',
        'description': 'Attacker creates fake login page to steal credentials',
        'risk': 'High',
        'mitigation': 'Entra ID authentication (federated, not custom login form). FIDO2/passkeys for phishing resistance.',
        'azure_feature': 'Entra External ID (CIAM) with phishing-resistant MFA',
    },
    {
        'element': 'User → Front Door (TB1)',
        'threat': 'Tampering',
        'description': 'Attacker manipulates HTTP requests (e.g., changes price in cart)',
        'risk': 'High',
        'mitigation': 'Server-side validation of all inputs. WAF on Front Door. Input sanitization.',
        'azure_feature': 'Azure WAF (OWASP 3.2 ruleset), input validation in API code',
    },
    {
        'element': 'AKS API',
        'threat': 'Repudiation',
        'description': 'Customer disputes a purchase, no proof of transaction',
        'risk': 'Medium',
        'mitigation': 'Comprehensive audit logging. Immutable transaction log. Digital receipt.',
        'azure_feature': 'Application Insights + Sentinel for transaction audit trail',
    },
    {
        'element': 'AKS → Azure SQL (TB2)',
        'threat': 'Information Disclosure',
        'description': 'SQL injection exposes customer PII and payment data',
        'risk': 'Critical',
        'mitigation': 'Parameterized queries (ORM). Encrypt sensitive columns. Private endpoint.',
        'azure_feature': 'Azure SQL Always Encrypted, Defender for SQL threat detection, Private Endpoint',
    },
    {
        'element': 'Front Door',
        'threat': 'Denial of Service',
        'description': 'DDoS attack during Black Friday sale takes store offline',
        'risk': 'High',
        'mitigation': 'DDoS protection, rate limiting, CDN caching, auto-scaling AKS.',
        'azure_feature': 'Azure DDoS Protection Standard + Front Door WAF rate limiting + AKS HPA',
    },
    {
        'element': 'AKS → Payment Gateway (TB3)',
        'threat': 'Elevation of Privilege',
        'description': 'Compromised container escalates to access payment gateway credentials',
        'risk': 'Critical',
        'mitigation': 'Workload identity (no stored credentials). Pod security standards. Network policies.',
        'azure_feature': 'AKS Workload Identity + Key Vault CSI + Pod Security Admission (restricted)',
    },
    {
        'element': 'Key Vault',
        'threat': 'Information Disclosure',
        'description': 'Attacker accesses stored secrets (DB connection strings, API keys)',
        'risk': 'Critical',
        'mitigation': 'Private endpoint. RBAC (not access policies). Managed identity access only. Audit logging.',
        'azure_feature': 'Key Vault with Private Endpoint, RBAC, Defender for Key Vault, diagnostic logs → Sentinel',
    },
]

print(APP_DESCRIPTION)
print('=== STRIDE Threat Analysis ===\n')
print(f'{"Element":<30} {"STRIDE":<12} {"Risk":<10} {"Azure Mitigation"}')
print('─' * 120)
for t in STRIDE_ANALYSIS:
    print(f'{t["element"]:<30} {t["threat"]:<12} {t["risk"]:<10} {t["azure_feature"]}')

## DevSecOps Pipeline Design

### Shift-left security: embed security at every stage of the pipeline

```
CODE          BUILD           TEST           DEPLOY          OPERATE
 │              │               │               │               │
 ▼              ▼               ▼               ▼               ▼
┌──────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
│Pre-   │    │ SAST     │    │ DAST     │    │ Infra    │    │ Runtime  │
│commit │    │ SCA      │    │ Pen test │    │ scanning │    │ protect  │
│hooks  │    │ Container│    │ Fuzz     │    │ Policy   │    │ Monitor  │
│       │    │ scan     │    │ testing  │    │ gates    │    │ Respond  │
└──────┘    └──────────┘    └──────────┘    └──────────┘    └──────────┘
 • Secrets   • Dependency    • API security   • IaC scanning  • Defender
   scanning    vulnerabilities • Auth testing    (Bicep/TF)   • Sentinel
 • Linting   • Code quality  • Injection       • Azure Policy • Incident
 • .gitignore• License check    testing          compliance     response
 • Credential• Image CVEs   • Business logic  • Approval      • Patching
   detection                    testing          gates
```

### Microsoft tools for each stage:

| Stage | Tool | Purpose |
|-------|------|---------|
| Code | GitHub Advanced Security (GHAS) | Secret scanning, code scanning (CodeQL) |
| Code | Credential Scanner (Azure DevOps) | Detect secrets in code |
| Build | Defender for DevOps | Connect GitHub/ADO to Defender for Cloud |
| Build | Defender for Containers | Image vulnerability scanning in ACR |
| Build | GitHub Dependabot / ADO Component Governance | SCA (dependency scanning) |
| Deploy | Azure Policy | Prevent non-compliant deployments |
| Deploy | Deployment gates (ADO) | Require security approval for prod |
| Operate | Defender for Cloud | Runtime protection |
| Operate | Microsoft Sentinel | Threat detection and response |

In [ ]:
# ===================================================================
# DEVSECOPS PIPELINE DESIGN EXERCISE
# Design security gates for a CI/CD pipeline
# ===================================================================

PIPELINE_SECURITY_GATES = [
    {
        'stage': 'Pre-commit',
        'gate': 'Secret detection',
        'tool': 'GitHub secret scanning push protection',
        'action': 'Block commit if secrets detected',
        'blocking': True,
    },
    {
        'stage': 'PR / Build',
        'gate': 'Static Application Security Testing (SAST)',
        'tool': 'CodeQL (via GitHub Advanced Security)',
        'action': 'Flag high/critical findings, require fix before merge',
        'blocking': True,
    },
    {
        'stage': 'PR / Build',
        'gate': 'Software Composition Analysis (SCA)',
        'tool': 'Dependabot / GitHub Dependency Review',
        'action': 'Block PR if new dependency has critical CVE',
        'blocking': True,
    },
    {
        'stage': 'PR / Build',
        'gate': 'Container image scan',
        'tool': 'Defender for Containers (ACR task)',
        'action': 'Scan image on push to ACR, quarantine if critical CVEs',
        'blocking': True,
    },
    {
        'stage': 'PR / Build',
        'gate': 'IaC scanning',
        'tool': 'Defender for DevOps (Bicep/Terraform scanning)',
        'action': 'Detect misconfigurations in infrastructure code',
        'blocking': False,
    },
    {
        'stage': 'Staging',
        'gate': 'DAST (Dynamic Application Security Testing)',
        'tool': 'OWASP ZAP or commercial DAST tool',
        'action': 'Run against staging environment, report findings',
        'blocking': False,
    },
    {
        'stage': 'Pre-production',
        'gate': 'Security approval gate',
        'tool': 'Azure DevOps / GitHub environment protection rules',
        'action': 'Security team must approve production deployments',
        'blocking': True,
    },
    {
        'stage': 'Production deploy',
        'gate': 'Azure Policy compliance check',
        'tool': 'Azure Policy (deny non-compliant resources)',
        'action': 'Deployment fails if resources violate policy',
        'blocking': True,
    },
]

print('=== DevSecOps Security Gates ===\n')
print(f'{"Stage":<18} {"Gate":<40} {"Tool":<40} {"Blocking?"}')
print('─' * 120)
for gate in PIPELINE_SECURITY_GATES:
    blocking = 'BLOCKING' if gate['blocking'] else 'Advisory'
    print(f'{gate["stage"]:<18} {gate["gate"]:<40} {gate["tool"]:<40} {blocking}')

blocking_count = sum(1 for g in PIPELINE_SECURITY_GATES if g['blocking'])
print(f'\nBlocking gates: {blocking_count}/{len(PIPELINE_SECURITY_GATES)}')
print('Design principle: Block on high-severity issues only. Advisory for medium/low to avoid developer friction.')

In [ ]:
# ===================================================================
# WORKLOAD IDENTITY DESIGN
# How applications authenticate to Azure services
# ===================================================================

WORKLOAD_IDENTITY_OPTIONS = [
    {
        'scenario': 'App Service accessing Azure SQL',
        'recommended': 'System-assigned managed identity',
        'why': 'Simplest, lifecycle tied to the app. No credential management.',
        'avoid': 'Connection string with SQL username/password',
    },
    {
        'scenario': 'Multiple apps accessing same Key Vault',
        'recommended': 'User-assigned managed identity (shared)',
        'why': 'One identity, one RBAC assignment. Survives app redeployment.',
        'avoid': 'Separate service principals with client secrets',
    },
    {
        'scenario': 'GitHub Actions deploying to Azure',
        'recommended': 'Workload identity federation (OIDC)',
        'why': 'No secrets stored in GitHub. GitHub token exchanged for Azure token.',
        'avoid': 'Service principal with client secret stored in GitHub Secrets',
    },
    {
        'scenario': 'AKS pods accessing Azure services',
        'recommended': 'AKS Workload Identity (federated)',
        'why': 'Pod-level identity. No shared node identity. Kubernetes SA projected token.',
        'avoid': 'Pod Identity (v1, deprecated) or shared AKS kubelet identity',
    },
    {
        'scenario': 'On-premises app accessing Azure',
        'recommended': 'Managed identity via Azure Arc (if possible) or certificate-based SP',
        'why': 'Arc-enabled servers get managed identities. Otherwise, certificate > secret.',
        'avoid': 'Service principal with long-lived client secret',
    },
    {
        'scenario': 'Multi-tenant SaaS app accessing customer tenants',
        'recommended': 'Multi-tenant app registration with admin consent',
        'why': 'Single app registration in your tenant, customers consent to access.',
        'avoid': 'Separate app registrations per customer',
    },
]

print('=== Workload Identity Design Guide ===\n')
for wi in WORKLOAD_IDENTITY_OPTIONS:
    print(f'\n--- {wi["scenario"]} ---')
    print(f'  Recommended: {wi["recommended"]}')
    print(f'  Why: {wi["why"]}')
    print(f'  Avoid: {wi["avoid"]}')

print('\n\nIdentity preference order (most to least secure):')
print('  1. Managed identity (system-assigned)')
print('  2. Managed identity (user-assigned)')
print('  3. Workload identity federation (OIDC)')
print('  4. Service principal with certificate')
print('  5. Service principal with client secret (LAST RESORT)')

In [ ]:
# ===================================================================
# API SECURITY ARCHITECTURE
# ===================================================================

API_SECURITY_DESIGN = {
    'Azure API Management (APIM)': {
        'role': 'API gateway — centralized security, rate limiting, authentication',
        'security_features': [
            'OAuth 2.0 / OpenID Connect validation (Entra ID)',
            'Subscription keys for API consumers',
            'Rate limiting and throttling policies',
            'IP filtering and geo-restriction',
            'Request/response transformation (strip sensitive headers)',
            'Certificate validation for mTLS',
            'WAF integration via Application Gateway',
        ],
    },
    'API authentication patterns': {
        'internal_apis': 'Managed identity (service-to-service within Azure)',
        'partner_apis': 'OAuth 2.0 client credentials flow with certificate',
        'customer_apis': 'OAuth 2.0 authorization code flow with PKCE',
        'webhook_apis': 'HMAC signature validation + IP allowlist',
    },
    'WAF deployment patterns': {
        'Global apps (multi-region)': 'Azure Front Door + WAF',
        'Regional apps + APIM': 'Application Gateway + WAF → APIM → Backend',
        'Simple web apps': 'Application Gateway + WAF → App Service',
        'APIs only': 'APIM policies (rate limit, IP filter, JWT validation)',
    },
}

print('=== API Security Architecture ===\n')
for category, details in API_SECURITY_DESIGN.items():
    print(f'\n{"=" * 70}')
    print(f'{category}')
    print(f'{"=" * 70}')
    if isinstance(details, dict):
        if 'role' in details:
            print(f'  Role: {details["role"]}')
            for feature in details['security_features']:
                print(f'  • {feature}')
        else:
            for key, value in details.items():
                print(f'  {key}: {value}')

In [ ]:
# ===================================================================
# APPLICATION SECURITY QUIZ
# ===================================================================

QUIZ = [
    {
        'question': 'During threat modeling of an API, you identify that an attacker could modify\n'
                    'the price field in an HTTP request body. Which STRIDE category is this?',
        'options': {
            'A': 'Spoofing',
            'B': 'Tampering',
            'C': 'Information Disclosure',
            'D': 'Elevation of Privilege',
        },
        'answer': 'B',
        'explanation': 'Tampering = modifying data without authorization. Changing a price in the request body '
                       'is data modification (integrity violation). The mitigation is server-side validation — '
                       'never trust client-provided prices, always look up from the database.',
    },
    {
        'question': 'A GitHub Actions workflow needs to deploy resources to Azure.\n'
                    'What is the MOST secure authentication method?',
        'options': {
            'A': 'Store a service principal client secret in GitHub Secrets',
            'B': 'Use workload identity federation (OIDC) with a managed identity',
            'C': 'Store an Azure CLI refresh token in GitHub Secrets',
            'D': 'Use a personal access token (PAT) for Azure DevOps',
        },
        'answer': 'B',
        'explanation': 'Workload identity federation uses OIDC to exchange the GitHub-issued JWT token for an '
                       'Azure AD token — no secrets stored in GitHub at all. The trust relationship is configured '
                       'in Azure AD (federated credential on app registration). Client secrets can leak and must '
                       'be rotated. PATs are user-bound and overly permissive.',
    },
    {
        'question': 'Tailspin Toys has a public-facing REST API and a web application.\n'
                    'Where should WAF be deployed for BEST protection?',
        'options': {
            'A': 'Azure Firewall in the hub VNet',
            'B': 'Azure Front Door with WAF policy',
            'C': 'NSG rules on the AKS subnet',
            'D': 'APIM rate limiting policies',
        },
        'answer': 'B',
        'explanation': 'Azure Front Door + WAF provides L7 protection at the edge — closest to the attacker. '
                       'It blocks OWASP Top 10 attacks (SQLi, XSS), provides bot protection, and handles DDoS '
                       'before traffic reaches your infrastructure. Azure Firewall is L4 (not web-aware). '
                       'NSGs are L3/L4. APIM rate limiting is complementary but not a WAF.',
    },
]

print('=== Application Security Architecture Quiz ===\n')
for i, q in enumerate(QUIZ, 1):
    print(f'Question {i}:')
    print(f'{q["question"]}\n')
    for key, option in q['options'].items():
        marker = '>>>' if key == q['answer'] else '   '
        print(f'  {marker} {key}. {option}')
    print(f'\n  Answer: {q["answer"]}')
    print(f'  Why: {q["explanation"]}')
    print()